# Supreme AI Judge: judge-calibrated BLEU

BLEU-4 only rewards exact n-gram overlap with the reference, so a candidate
summary that is factually identical to the reference but phrased with
different (but synonymous) words gets penalized for wording, not for being
wrong.

This notebook builds a metric that tries to strip out *that* penalty without
opening the door to a model just rewriting the candidate to match the
reference verbatim:

1. Compute the candidate's raw BLEU-4 against the reference.
2. Show a judge LLM the reference, the candidate, and that raw BLEU-4 score,
   along with an explanation of BLEU-4's synonym-blindness.
3. Ask the judge to produce a **minimally revised candidate** that may only
   swap in reference wording for genuinely close-in-meaning terms (e.g.
   `connect` / `connects` / `connection`) -- never for terms that differ in
   meaning (e.g. `send` vs. `publish` describe different actions and must not
   be swapped), and never by adding information the candidate didn't already
   contain.
4. Re-score the revised candidate with BLEU-4. That's the **judge-calibrated
   BLEU**: how good the candidate would have scored if its raw BLEU-4 hadn't
   also been penalizing harmless wording choices.

The gap between raw BLEU and judge-calibrated BLEU tells us how much of a
candidate's BLEU deficit is purely a wording artifact vs. a real content gap.

## Config

In [14]:
# ── Config ──────────────────────────────────────────────────────────────────
REPO_ROOT = "/home/phuc/code-sum"

RUN_IDS = {
    "few_shot_llm": "8ed48f980c5540e8b029056b268867e8",
    "few_shot_all_context": "7e938318a5974bcaaf0c424d43bae43e",
    "zero_shot": "5987585716d54933b79acdbedaaaad15",
    "metagente": "512522dd8ba14ff4b8930906cea34a0f",
}

JUDGE_MODEL = "openai/gpt-4o-mini"
JUDGE_BACKEND = "openrouter"
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 300
MAX_CONCURRENCY = 10

N_SAMPLES = 30  # sampled per run
RANDOM_SEED = 0

In [15]:
import asyncio
import math
import pathlib
import random
import re
import sys
import xml.sax.saxutils

import mlflow
import pandas as pd
from mlflow import MlflowClient
from tqdm.asyncio import tqdm as atqdm

sys.path.insert(0, str(pathlib.Path(REPO_ROOT) / "src"))
from mas_code_sum.methods.llm_client import make_clients

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

## Load predictions from MLflow

Same pattern as `llm_judge_prompts.ipynb`: each run logged a
`predictions/*.csv` artifact with columns `id, project, func_name, run,
reference, prediction`. `id` indexes into `dataset/full/test.jsonl`.

In [16]:
def load_predictions(run_name: str, run_id: str) -> pd.DataFrame:
    local_dir = client.download_artifacts(run_id, "predictions")
    csvs = list(pathlib.Path(local_dir).glob("*.csv"))
    assert len(csvs) == 1, f"expected exactly one predictions csv, found {csvs}"
    df = pd.read_csv(csvs[0])
    df["system"] = run_name
    return df


pred_dfs = {name: load_predictions(name, run_id) for name, run_id in RUN_IDS.items()}
for name, df in pred_dfs.items():
    print(name, df.shape)

few_shot_llm (200, 7)
few_shot_all_context (200, 7)
zero_shot (200, 7)
metagente (200, 7)


In [17]:
random.seed(RANDOM_SEED)

sampled_rows = []
for name, df in pred_dfs.items():
    n = min(N_SAMPLES, len(df))
    sampled_rows.append(df.sample(n=n, random_state=RANDOM_SEED))

judge_input_df = pd.concat(sampled_rows, ignore_index=True)
print(f"{len(judge_input_df)} (system, sample) rows to calibrate")
judge_input_df.head()

120 (system, sample) rows to calibrate


,id,project,func_name,run,reference,prediction,system
0,218,apache/airflow,DagRun.get_previous_dagrun,0,The previous DagRun if there is one,Get the previous DagRun for this DAG .,few_shot_llm
1,1820,tamboui/tamboui,Monthly.of,0,Creates a calendar for the month containing th...,Here's a summary of the provided code snippets:,few_shot_llm
2,1607,newton-physics/newton,ViewerBase._qualify,0,Prefix a backend object name with the active l...,Qualify a name with the active layer's prefix .,few_shot_llm
3,548,orientechnologies/orientdb,ONodeManager.initCheckLeader,0,init the procedure that sends pings to other s...,Initializes the check leader thread.,few_shot_llm
4,1827,tamboui/tamboui,ApiLevelsSnippets.inlineTuiRunnerBasic,0,========== INLINE TUI RUNNER ==========,Runs the demo application.,few_shot_llm


## BLEU-4

Smoothed corpus-style BLEU-4 for a single (reference, candidate) pair,
inlined so this notebook stays import-free of the eval harness (same formula
as `mas_code_sum.evaluator.bleu`, and as `llm_judge_prompts.ipynb`).

In [18]:
_normalize1 = [
    (re.compile(r"<skipped>"), ""),
    (re.compile(r"-\n"), ""),
    (re.compile(r"\n"), " "),
]
_normalize2 = [
    (re.compile(r"([\{-\~\[-\` -\&\(-\+\:-\@\/])"), r" \1 "),
    (re.compile(r"([^0-9])([\.,])"), r"\1 \2 "),
    (re.compile(r"([\.,])([^0-9])"), r" \1 \2"),
    (re.compile(r"([0-9])(-)"), r"\1 \2 "),
]


def _normalize(s):
    if not isinstance(s, str):
        s = " ".join(s)
    for pattern, replace in _normalize1:
        s = pattern.sub(replace, s)
    s = xml.sax.saxutils.unescape(s, {"&quot;": '"'})
    s = " %s " % s
    s = s.lower()
    for pattern, replace in _normalize2:
        s = pattern.sub(replace, s)
    return s.split()


def _count_ngrams(words, n=4):
    counts = {}
    for k in range(1, n + 1):
        for i in range(len(words) - k + 1):
            ngram = tuple(words[i : i + k])
            counts[ngram] = counts.get(ngram, 0) + 1
    return counts


def _cook_refs(refs, n=4):
    refs = [_normalize(ref) for ref in refs]
    maxcounts = {}
    for ref in refs:
        counts = _count_ngrams(ref, n)
        for ngram, count in counts.items():
            maxcounts[ngram] = max(maxcounts.get(ngram, 0), count)
    return [len(ref) for ref in refs], maxcounts


def _cook_test(test, item, n=4):
    reflens, refmaxcounts = item
    test = _normalize(test)
    result = {"testlen": len(test), "reflen": min(reflens)}
    result["guess"] = [max(len(test) - k + 1, 0) for k in range(1, n + 1)]
    result["correct"] = [0] * n
    counts = _count_ngrams(test, n)
    for ngram, count in counts.items():
        result["correct"][len(ngram) - 1] += min(refmaxcounts.get(ngram, 0), count)
    return result


def _score_cooked(comps, n=4):
    logbleu = 0.0
    for k in range(n):
        correct = comps["correct"][k]
        guess = comps["guess"][k]
        addsmooth = 1 if k > 0 else 0
        logbleu += math.log(correct + addsmooth + sys.float_info.min) - math.log(guess + addsmooth + sys.float_info.min)
    logbleu /= float(n)
    brev_penalty = min(0, 1 - float(comps["reflen"] + 1) / (comps["testlen"] + 1))
    return math.exp(logbleu + brev_penalty)


def sentence_bleu(reference: str, candidate: str) -> float:
    '''Smoothed corpus-style BLEU-4 for a single (reference, candidate) pair, 0-100 scale.'''
    item = _cook_refs([reference])
    comps = _cook_test(candidate, item)
    return _score_cooked(comps) * 100

In [19]:
judge_input_df["bleu"] = judge_input_df.apply(
    lambda row: sentence_bleu(row["reference"], row["prediction"]), axis=1
)
judge_input_df[["system", "id", "reference", "prediction", "bleu"]].head()

,system,id,reference,prediction,bleu
0,few_shot_llm,218,The previous DagRun if there is one,Get the previous DagRun for this DAG .,2.860624e+01
1,few_shot_llm,1820,Creates a calendar for the month containing th...,Here's a summary of the provided code snippets:,8.789051e+00
2,few_shot_llm,1607,Prefix a backend object name with the active l...,Qualify a name with the active layer's prefix .,3.249905e+01
3,few_shot_llm,548,init the procedure that sends pings to other s...,Initializes the check leader thread.,4.769377e+00
4,few_shot_llm,1827,========== INLINE TUI RUNNER ==========,Runs the demo application.,1.461074e-77


## The judge proposes edits, it doesn't rewrite

Letting the judge hand back a freely rewritten sentence is exactly the
cheating hole: nothing stops it from quietly copying reference phrasing that
isn't actually a synonym, or padding in content that wasn't in the candidate.

Instead the judge may only emit a **JSON list of edit operations** from a
fixed, narrow vocabulary. We apply those operations ourselves with plain
string code -- the judge never touches the text directly, so it's structurally
incapable of doing anything outside the allowed operation set, and we
independently re-validate every operation before applying it.

In [20]:
EDIT_PROMPT = """You are proposing a small set of mechanical edits to a \
candidate summary so it aligns more closely with a reference summary's \
wording, for a metric-calibration exercise. You do NOT rewrite the text \
yourself -- you may only propose edits from the fixed operation list below, \
and each one is applied programmatically afterward, so every edit must be \
literal, unambiguous, and reference an exact word that appears in the \
candidate.

Reference summary:
{reference}

Candidate summary:
{prediction}

Allowed operations (respond with a JSON array containing zero or more of \
these objects -- nothing else):

1. {{"op": "replace", "from": "<word in candidate>", "to": "<word>"}} -- \
ONLY for a strict grammatical/morphological variant of the SAME word: tense, \
number, or part-of-speech inflection of one shared stem. Examples of VALID \
replacements: "connect" -> "connections", "collect" -> "collecting", \
"runs" -> "run". Examples of INVALID replacements (different words, do NOT \
propose): "send" -> "publish", "get" -> "retrieve" -- these are different \
words that merely mean something similar, not inflections of one word.
2. {{"op": "replace", "from": "<article>", "to": "<article>"}} -- replace one \
article with another. "from" and "to" must each be exactly one of "a", "an", \
"the".
3. {{"op": "remove", "text": "<single word>"}} -- delete one occurrence of a \
single word from the candidate.
4. {{"op": "remove", "text": "<single character>"}} -- delete one occurrence \
of a single character from the candidate (e.g. dropping a trailing "s").

Do not propose anything else: no reordering, no adding words, no replacing a \
word with a different word that merely means something similar. Propose the \
smallest set of edits (may be empty) that would raise word-overlap with the \
reference without changing what the candidate claims. If nothing valid \
applies, respond with [].

Respond with ONLY the JSON array, no commentary, no markdown code fences.
"""

## Applying edits programmatically

The judge's JSON is *proposed* edits -- we independently re-validate every one
before applying it, so a non-compliant edit (e.g. a meaning-changing word
swap) can't sneak through even if the judge ignores instructions.

- `replace`: allowed only if `from`/`to` are both articles (`a`/`an`/`the`),
  or share a common stem (heuristic: their lowercase forms agree on a prefix
  covering all but the last couple characters) -- a cheap proxy for "same
  word, different inflection" that rejects unrelated words like
  `send` -> `publish`.
- `remove`: a single word (whole-word match) or a single character
  (substring match), first occurrence only.

Edits are applied in the order given, each to the output of the previous
one.

In [21]:
ARTICLES = {"a", "an", "the"}


def _is_valid_replacement(frm: str, to: str) -> bool:
    frm_l, to_l = frm.lower(), to.lower()
    if frm_l in ARTICLES and to_l in ARTICLES:
        return True
    if not frm_l.isalpha() or not to_l.isalpha():
        return False
    if frm_l == to_l:
        return False
    stem_len = max(3, min(len(frm_l), len(to_l)) - 2)
    return frm_l[:stem_len] == to_l[:stem_len]


def _replace_first_word(text: str, frm: str, to: str) -> tuple[str, bool]:
    pattern = re.compile(rf"\b{re.escape(frm)}\b", re.IGNORECASE)
    match = pattern.search(text)
    if not match:
        return text, False
    matched = match.group(0)
    replacement = to[0].upper() + to[1:] if matched[:1].isupper() else to
    return text[: match.start()] + replacement + text[match.end() :], True


def _remove_first_word(text: str, word: str) -> tuple[str, bool]:
    pattern = re.compile(rf"\s?\b{re.escape(word)}\b", re.IGNORECASE)
    match = pattern.search(text)
    if not match:
        return text, False
    return text[: match.start()] + text[match.end() :], True


def _remove_first_char(text: str, char: str) -> tuple[str, bool]:
    idx = text.find(char)
    if idx == -1:
        return text, False
    return text[:idx] + text[idx + 1 :], True


def apply_edits(text: str, edits: list[dict]) -> tuple[str, list[dict]]:
    '''Apply a list of judge-proposed edit ops to *text*, re-validating each.'''
    log = []
    for edit in edits:
        op = edit.get("op")
        if op == "replace":
            frm, to = edit.get("from", ""), edit.get("to", "")
            if not _is_valid_replacement(frm, to):
                log.append({**edit, "applied": False, "reason": "not an article pair or morphological variant"})
                continue
            new_text, applied = _replace_first_word(text, frm, to)
            log.append({**edit, "applied": applied, "reason": None if applied else "word not found"})
            if applied:
                text = new_text
        elif op == "remove":
            value = edit.get("text", "")
            if not value:
                log.append({**edit, "applied": False, "reason": "empty text"})
                continue
            if len(value) == 1:
                new_text, applied = _remove_first_char(text, value)
            else:
                new_text, applied = _remove_first_word(text, value)
            log.append({**edit, "applied": applied, "reason": None if applied else "not found"})
            if applied:
                text = new_text
        else:
            log.append({**edit, "applied": False, "reason": f"unknown op {op!r}"})
    return text, log

## Run the judge and apply its proposed edits

In [22]:
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.MULTILINE)


def parse_edit_list(judge_output: str) -> list[dict]:
    cleaned = _FENCE_RE.sub("", judge_output).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []


_, async_client = make_clients(JUDGE_BACKEND)
_sem = asyncio.Semaphore(MAX_CONCURRENCY)


async def calibrate_one(row: pd.Series) -> dict:
    prompt = EDIT_PROMPT.format(reference=row["reference"], prediction=row["prediction"])
    async with _sem:
        response = await async_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=JUDGE_TEMPERATURE,
            max_tokens=JUDGE_MAX_TOKENS,
        )
    text = response.choices[0].message.content
    proposed_edits = parse_edit_list(text)
    revised, edit_log = apply_edits(row["prediction"], proposed_edits)
    return {"judge_output": text, "proposed_edits": proposed_edits, "edit_log": edit_log, "revised_prediction": revised}

In [23]:
import json


async def run_calibration(df: pd.DataFrame) -> pd.DataFrame:
    tasks = [calibrate_one(row) for _, row in df.iterrows()]
    results = await atqdm.gather(*tasks, desc="calibrating")
    return pd.DataFrame(results)


_calibration_results = await run_calibration(judge_input_df)
calibration_df = pd.concat(
    [judge_input_df.reset_index(drop=True), _calibration_results.reset_index(drop=True)], axis=1
)
calibration_df[["system", "id", "reference", "prediction", "revised_prediction", "bleu"]].head()

calibrating: 100%|██████████| 120/120 [00:16<00:00,  7.13it/s]


,system,id,reference,prediction,revised_prediction,bleu
0,few_shot_llm,218,The previous DagRun if there is one,Get the previous DagRun for this DAG .,Get the previous DagRun for DAG .,2.860624e+01
1,few_shot_llm,1820,Creates a calendar for the month containing th...,Here's a summary of the provided code snippets:,Here's a summary of the provided code snippets:,8.789051e+00
2,few_shot_llm,1607,Prefix a backend object name with the active l...,Qualify a name with the active layer's prefix .,Qualify a name with the active layer's prefix .,3.249905e+01
3,few_shot_llm,548,init the procedure that sends pings to other s...,Initializes the check leader thread.,Initializes the check leader thread.,4.769377e+00
4,few_shot_llm,1827,========== INLINE TUI RUNNER ==========,Runs the demo application.,Runs the demo application.,1.461074e-77


## Judge-calibrated BLEU

Re-score the *programmatically edited* candidate against the same reference.
This is the judge-calibrated BLEU.

In [24]:
calibration_df["calibrated_bleu"] = calibration_df.apply(
    lambda row: sentence_bleu(row["reference"], row["revised_prediction"]), axis=1
)
calibration_df["bleu_delta"] = calibration_df["calibrated_bleu"] - calibration_df["bleu"]
calibration_df[["system", "id", "bleu", "calibrated_bleu", "bleu_delta"]].sort_values(
    "bleu_delta", ascending=False
).head(10)

,system,id,bleu,calibrated_bleu,bleu_delta
22,few_shot_llm,245,25.848658,60.427508,34.578850
54,few_shot_all_context,510,33.265097,49.492320,16.227223
110,metagente,1833,15.320779,30.213754,14.892975
87,zero_shot,1835,9.941491,16.451929,6.510438
45,few_shot_all_context,207,30.058408,35.930411,5.872003
105,metagente,207,14.882560,20.577843,5.695282
114,metagente,510,11.571771,16.853999,5.282228
86,zero_shot,1829,11.678449,16.463248,4.784799
30,few_shot_all_context,218,28.606242,33.265097,4.658855
0,few_shot_llm,218,28.606242,33.265097,4.658855


## Aggregate: raw vs. judge-calibrated BLEU, per system

In [25]:
system_summary = calibration_df.groupby("system").agg(
    mean_bleu=("bleu", "mean"),
    mean_calibrated_bleu=("calibrated_bleu", "mean"),
    mean_delta=("bleu_delta", "mean"),
    n=("bleu", "size"),
)
system_summary

,mean_bleu,mean_calibrated_bleu,mean_delta,n
system,,,,
few_shot_all_context,24.326172,25.290478,0.964306,30
few_shot_llm,18.650775,19.293064,0.642290,30
metagente,12.851975,13.753766,0.901791,30
zero_shot,13.805088,14.579983,0.774895,30


## Spot-check

Every edit here was applied by plain string code after independent
re-validation, so this is just eyeballing that the *proposed* edits were
sensible -- not checking for smuggled content, since the pipeline structurally
can't add any.

In [26]:
for _, row in calibration_df.sort_values("bleu_delta", ascending=False).head(5).iterrows():
    print(f"=== system={row['system']} id={row['id']}  bleu {row['bleu']:.1f} -> {row['calibrated_bleu']:.1f} (Δ{row['bleu_delta']:+.1f}) ===")
    print("reference:         ", row["reference"])
    print("prediction:        ", row["prediction"])
    print("revised prediction:", row["revised_prediction"])
    print("edit log:          ", row["edit_log"])
    print()

=== system=few_shot_llm id=245  bleu 25.8 -> 60.4 (Δ+34.6) ===
reference:          Returns the DagRun for this TaskInstance
prediction:         Get the DagRun object for this DAG .
revised prediction: Get the DagRun for this .
edit log:           [{'op': 'replace', 'from': 'Get', 'to': 'Returns', 'applied': False, 'reason': 'not an article pair or morphological variant'}, {'op': 'remove', 'text': 'object', 'applied': True, 'reason': None}, {'op': 'remove', 'text': 'DAG', 'applied': True, 'reason': None}]

=== system=few_shot_all_context id=510  bleu 33.3 -> 49.5 (Δ+16.2) ===
reference:          Disconnects a client connections
prediction:         Disconnects a client connection by id.
revised prediction: Disconnects a client connection.
edit log:           [{'op': 'remove', 'text': 'by', 'applied': True, 'reason': None}, {'op': 'remove', 'text': 'id', 'applied': True, 'reason': None}]

=== system=metagente id=1833  bleu 15.3 -> 30.2 (Δ+14.9) ===
reference:          Creates a new defaul

In [27]:
# How many proposed edits were accepted vs. rejected by our own re-validation?
all_edits = [e for log in calibration_df["edit_log"] for e in log]
edits_df = pd.DataFrame(all_edits)
print(f"{len(all_edits)} edits proposed across {len(calibration_df)} rows")
edits_df["applied"].value_counts() if not edits_df.empty else "no edits proposed" 

155 edits proposed across 120 rows


applied
True     115
False     40
Name: count, dtype: int64